In [ ]:
# Install required packages
import sys
import subprocess

packages = [
    "pandas", "numpy", "torch", "transformers",
    "scikit-learn", "tqdm", "sentencepiece", "sacremoses"
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True
    )
    status = "OK" if result.returncode == 0 else f"FAIL: {result.stderr[:80]}"
    print(f"{pkg:<20} {status}")

In [ ]:
import os
import json
import zipfile
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm
import shutil
warnings.filterwarnings('ignore')

# Device setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device  :", DEVICE)
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("Torch   :", torch.__version__)

# Disk check
total, used, free = shutil.disk_usage('/')
print(f"Disk    : {free // (1024**3)} GB free / {total // (1024**3)} GB total")

In [ ]:
# Load all data files
train       = pd.read_csv('train_data_SMM4H_2026_Task_1.csv')
dev         = pd.read_csv('dev_data_SMM4H_2026_Task_1.csv')
train_cadec = pd.read_csv('train_data_cadec_translated.csv')
dev_cadec   = pd.read_csv('dev_data_cadec_translated.csv')
test        = pd.read_csv('combined_test_data_unlabeled.csv')

# Combine datasets
train_full = pd.concat([train, train_cadec], ignore_index=True)
dev_full   = pd.concat([dev, dev_cadec], ignore_index=True)
train_final = pd.concat([train_full, dev_full], ignore_index=True)

# Clean labels
train_full['label'] = train_full['label'].astype(int)
dev_full['label']   = dev_full['label'].astype(int)
train_final['label'] = train_final['label'].astype(int)

# Normalize test language tags
test['language'] = test['language'].replace({'de_cadec': 'de', 'fr_cadec': 'fr'})

# Clean text
def clean_text(text):
    text = str(text).strip()
    return '[EMPTY]' if text.lower() in ['nan', 'none', ''] else text

for df in [train_full, dev_full, train_final, test]:
    df['text'] = df['text'].apply(clean_text)

print(f"train_full  : {train_full.shape}")
print(f"dev_full    : {dev_full.shape}")
print(f"train_final : {train_final.shape}")
print(f"test        : {test.shape}")

In [ ]:
# Configuration
CFG = {
    'model_name'      : 'xlm-roberta-large',
    'max_len'         : 128,
    'batch_size'      : 32,
    'grad_accum'      : 2,
    'epochs'          : 8,
    'lr'              : 2e-5,
    'warmup_ratio'    : 0.1,
    'focal_gamma'     : 2.0,
    'focal_alpha'     : 0.25,
    'seed'            : 42,
    'save_path'       : 'best_model.pt',
}

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(CFG['seed'])
print("Config ready.")

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce     = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none')
        pt      = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - pt) ** self.gamma * bce).mean()


class ADEDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['label'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


class ADEClassifier(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        hidden          = self.encoder.config.hidden_size
        self.drop       = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        out   = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls   = self.drop(out.last_hidden_state[:, 0, :])
        return self.classifier(cls).squeeze(-1)

print("Model classes defined.")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

# Language-balanced weighted sampler
sample_weights = np.zeros(len(train_final))
for lang in train_final['language'].unique():
    mask  = (train_final['language'] == lang).values
    n_pos = (train_final.loc[mask, 'label'] == 1).sum()
    n_neg = (train_final.loc[mask, 'label'] == 0).sum()
    n_tot = mask.sum()
    w_pos = n_tot / (2 * n_pos) if n_pos > 0 else 1.0
    w_neg = n_tot / (2 * n_neg) if n_neg > 0 else 1.0
    sample_weights[mask & (train_final['label'] == 1).values] = w_pos
    sample_weights[mask & (train_final['label'] == 0).values] = w_neg

sampler = WeightedRandomSampler(
    weights     = torch.tensor(sample_weights, dtype=torch.float),
    num_samples = len(train_final),
    replacement = True
)

train_dataset = ADEDataset(
    texts     = train_final['text'].tolist(),
    labels    = train_final['label'].tolist(),
    tokenizer = tokenizer,
    max_len   = CFG['max_len']
)

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                          sampler=sampler, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")

In [ ]:
set_seed(CFG['seed'])

model     = ADEClassifier(CFG['model_name']).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=0.01)

total_steps  = (len(train_loader) // CFG['grad_accum']) * CFG['epochs']
warmup_steps = int(total_steps * CFG['warmup_ratio'])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

criterion = FocalLoss(gamma=CFG['focal_gamma'], alpha=CFG['focal_alpha'])
scaler    = torch.cuda.amp.GradScaler()

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Total steps : {total_steps}")
print(f"Warmup steps: {warmup_steps}")

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='Train')):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        labs  = batch['label'].to(DEVICE)
        with torch.cuda.amp.autocast():
            loss = criterion(model(ids, mask), labs) / CFG['grad_accum']
        scaler.scale(loss).backward()
        total_loss += loss.item() * CFG['grad_accum']
        if (step + 1) % CFG['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
    return total_loss / len(loader)


def get_logits(model, loader):
    model.eval()
    logits_all = []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Eval'):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(ids, mask)
            logits_all.append(logits.cpu())
    return torch.cat(logits_all).numpy()

print("Training functions defined.")

In [ ]:
best_loss = float('inf')

for epoch in range(CFG['epochs']):
    print(f"\nEpoch {epoch+1}/{CFG['epochs']}")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, scaler)
    print(f"Train loss: {train_loss:.4f}")
    
    if train_loss < best_loss:
        best_loss = train_loss
        torch.save({'epoch': epoch, 'model_state': model.state_dict()}, CFG['save_path'])
        print(f"  --> Saved (loss={best_loss:.4f})")
    
    _, _, free = shutil.disk_usage('/')
    print(f"  Disk free: {free // (1024**3)} GB")

print("\nTraining complete!")

In [ ]:
# Load best model
checkpoint = torch.load(CFG['save_path'], map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
print(f"Loaded epoch {checkpoint['epoch']+1}")

# Prepare test data
test_reset = test.reset_index(drop=True)
test_dataset = ADEDataset(
    texts     = test_reset['text'].tolist(),
    labels    = None,
    tokenizer = tokenizer,
    max_len   = CFG['max_len']
)
test_loader = DataLoader(test_dataset, batch_size=CFG['batch_size']*2,
                         shuffle=False, num_workers=4, pin_memory=True)

# Get predictions
test_logits = get_logits(model, test_loader)
test_probs  = torch.sigmoid(torch.tensor(test_logits)).numpy()

# Save probabilities
np.save('test_probs.npy',  test_probs)
np.save('test_ids.npy',    test_reset['id'].values)
np.save('test_langs.npy',  test_reset['language'].values)
print("Probabilities saved.")

In [ ]:
# V3 thresholds (best submission)
THRESHOLDS = {
    'de' : 0.40,
    'en' : 0.10,
    'fr' : 0.25,
    'ja' : 0.10,
    'ru' : 0.10,
    'zh' : 0.06,
    'fa' : 0.50,
}

test_preds = np.zeros(len(test_reset), dtype=int)
for lang, thr in THRESHOLDS.items():
    mask  = test_reset['language'].values == lang
    test_preds[mask] = (test_probs[mask] >= thr).astype(int)

print("Per-language predictions:")
for lang, thr in THRESHOLDS.items():
    mask  = test_reset['language'].values == lang
    pos   = test_preds[mask].sum()
    total = mask.sum()
    print(f"{lang}: {pos}/{total} ({pos/total*100:.1f}%)")

print(f"\nTotal positives: {test_preds.sum()}")

In [ ]:
submission = pd.DataFrame({
    'id'              : test_reset['id'].values,
    'predicted_label' : test_preds
})

submission.to_csv('submission.csv', index=False)

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('submission.csv', 'submission.csv')

print(f"Submission ready: {len(submission)} rows")
print(f"Label distribution: {submission['predicted_label'].value_counts().to_dict()}")